### Preparación del Entorno

In [0]:
# Cargamos la tabla limpia y la convertimos en una Vista Temporal de Spark
df_citas = spark.table("default.citas_pmm_limpioV3")
df_citas.createOrReplaceTempView("v_citas_pmm")

print("Vista 'v_citas_pmm' creada exitosamente. Lista para análisis SQL.")

Vista 'v_citas_pmm' creada exitosamente. Lista para análisis SQL.


### N1: Rendimiento Financiero y Distribución de Cobertura por Aseguradora

In [0]:
%sql
CREATE OR REPLACE TABLE rendimiento_aseguradora AS
SELECT
    nom_sucursal AS Sucursal,
    especialidad_medica AS Especialidad,
    nom_aseguradora AS Aseguradora,
    COUNT(*) AS Volumen_Citas,
    ROUND(SUM(pago_aseg), 2) AS Total_Pagado_Aseguradora_USD,
    ROUND(SUM(pago_clie), 2) AS Total_Copago_Paciente_USD,
    ROUND(SUM(pago_total), 2) AS Ingresos_Totales_USD,
    ROUND((SUM(pago_aseg) / SUM(pago_total)) * 100, 2) AS Porcentaje_Cobertura_Promedio
FROM v_citas_pmm
WHERE estado_cita = 'Completada'
GROUP BY nom_aseguradora, especialidad_medica, nom_sucursal;

SELECT * FROM rendimiento_aseguradora;

Sucursal,Especialidad,Aseguradora,Volumen_Citas,Total_Pagado_Aseguradora_USD,Total_Copago_Paciente_USD,Ingresos_Totales_USD,Porcentaje_Cobertura_Promedio
PMM Costa del Este,Neurología,Particular / Sin Seguro,27,0.0,3506.63,3506.63,0.0
PMM Costa del Este,Neurología,PALIG,14,1160.0,510.75,1670.75,69.43
PMM Brisas del Golf,Neurología,Compañía Internacional de Seguros (IS),8,576.02,250.0,826.02,69.73
PMM El Dorado,Geriatría,MAPFRE Panamá,3,177.23,43.01,220.24,80.47
PMM San Francisco,Fisioterapia,PALIG,31,1292.67,616.46,1909.13,67.71
PMM El Dorado,Cardiología,Seguros SURA,11,780.21,301.64,1081.85,72.12
PMM El Dorado,Odontología,Seguros ASSA,30,1080.27,502.96,1583.23,68.23
PMM San Francisco,Radiología,Particular / Sin Seguro,45,0.0,4101.61,4101.61,0.0
PMM Tocumen,Nefrología,Seguros SURA,9,512.4,191.38,703.78,72.81
PMM Tocumen,Odontología,Particular / Sin Seguro,18,0.0,641.21,641.21,0.0


### N2: Tasa de Cancelación por Sucursal

In [0]:
%sql
CREATE OR REPLACE TABLE cancelacion_sucursal AS
SELECT 
    nom_sucursal AS Sucursal,
    especialidad_medica AS Especialidad,
    COUNT(*) AS Citas_Agendadas,
    SUM(CASE WHEN estado_cita = 'Cancelada' THEN 1 ELSE 0 END) AS Citas_Canceladas,
    SUM(CASE WHEN estado_cita = 'Completada' THEN 1 ELSE 0 END) AS Citas_Completadas,
    ROUND((SUM(CASE WHEN estado_cita = 'Cancelada' THEN 1 ELSE 0 END) / COUNT(*)) * 100, 2) AS Tasa_Cancelacion_PCT
FROM v_citas_pmm
GROUP BY nom_sucursal, especialidad_medica;
SELECT * FROM cancelacion_sucursal;

Sucursal,Especialidad,Citas_Agendadas,Citas_Canceladas,Citas_Completadas,Tasa_Cancelacion_PCT
PMM San Francisco,Odontología,277,54,223,19.49
PMM El Dorado,Cardiología,85,17,68,20.0
PMM Costa del Este,Otorrinolaringología,225,34,191,15.11
PMM Costa del Este,Oftalmología,227,43,184,18.94
PMM El Dorado,Geriatría,28,5,23,17.86
PMM San Francisco,Neurología,176,32,144,18.18
PMM Tocumen,Radiología,79,19,60,24.05
PMM Costa del Este,Ginecología,70,12,58,17.14
PMM Brisas del Golf,Pediatría,29,5,24,17.24
PMM El Dorado,Urología,74,14,60,18.92


### N3: Segmentación Demográfica y Ticket Promedio

In [0]:
%sql
CREATE OR REPLACE TABLE segmentacion_demografica AS
SELECT 
    nom_sucursal AS Sucursal,
    especialidad_medica AS Especialidad,
    CASE 
        WHEN edad_pac_cita < 15 THEN '1. Pediátrico (<15 años)'
        WHEN edad_pac_cita BETWEEN 15 AND 64 THEN '2. Adulto (15-64 años)'
        ELSE '3. Geriátrico (65+ años)' 
    END AS Segmento_Poblacional,
    COUNT(*) AS Total_Atenciones,
    ROUND(SUM(pago_total), 2) AS Facturacion_Total_USD,
    ROUND(AVG(pago_total), 2) AS Ticket_Promedio_USD
FROM v_citas_pmm
WHERE estado_cita = 'Completada'
GROUP BY 
    CASE 
        WHEN edad_pac_cita < 15 THEN '1. Pediátrico (<15 años)'
        WHEN edad_pac_cita BETWEEN 15 AND 64 THEN '2. Adulto (15-64 años)'
        ELSE '3. Geriátrico (65+ años)' 
    END, 
    nom_sucursal,
    especialidad_medica;
SELECT * FROM segmentacion_demografica;

Sucursal,Especialidad,Segmento_Poblacional,Total_Atenciones,Facturacion_Total_USD,Ticket_Promedio_USD
PMM Costa del Este,Odontología,1. Pediátrico (<15 años),53,3354.61,63.29
PMM Brisas del Golf,Cardiología,3. Geriátrico (65+ años),13,1248.07,96.01
PMM Tocumen,Psiquiatría,2. Adulto (15-64 años),44,2858.94,64.98
PMM Costa del Este,Dermatología,3. Geriátrico (65+ años),29,2888.99,99.62
PMM Tocumen,Nefrología,2. Adulto (15-64 años),38,3023.84,79.57
PMM Brisas del Golf,Dermatología,2. Adulto (15-64 años),70,4278.81,61.13
PMM El Dorado,Psiquiatría,2. Adulto (15-64 años),68,5296.59,77.89
PMM Tocumen,Pediatría,1. Pediátrico (<15 años),22,944.52,42.93
PMM Tocumen,Neurología,3. Geriátrico (65+ años),9,825.26,91.7
PMM San Francisco,Oftalmología,3. Geriátrico (65+ años),39,2840.61,72.84


### N4: Rentabilidad de cada Especialidad por Hora

In [0]:
%sql
CREATE OR REPLACE TABLE rentabilidad_especialidad AS
SELECT 
    nom_sucursal AS Sucursal,
    especialidad_medica AS Especialidad,
    COUNT(*) AS Total_Consultas,
    ROUND(SUM(mins_cit) / 60, 2) AS Horas_Totales_Invertidas,
    ROUND(SUM(pago_total), 2) AS Ingresos_Totales_USD,
    ROUND(SUM(pago_total) / (SUM(mins_cit) / 60), 2) AS Rentabilidad_Por_Hora_USD
FROM v_citas_pmm
WHERE estado_cita = 'Completada'
GROUP BY especialidad_medica, nom_sucursal;
SELECT * FROM rentabilidad_especialidad;

Sucursal,Especialidad,Total_Consultas,Horas_Totales_Invertidas,Ingresos_Totales_USD,Rentabilidad_Por_Hora_USD
PMM San Francisco,Radiología,215,139.0,19421.82,139.73
PMM Costa del Este,Pediatría,55,31.25,3652.32,116.87
PMM San Francisco,Urología,88,53.0,6045.62,114.07
PMM El Dorado,Otorrinolaringología,140,92.75,9794.89,105.61
PMM San Francisco,Gastroenterología,127,86.0,13741.03,159.78
PMM San Francisco,Nefrología,123,72.0,13484.3,187.28
PMM Tocumen,Otorrinolaringología,73,46.75,4330.98,92.64
PMM Brisas del Golf,Cardiología,49,27.0,4376.37,162.09
PMM Brisas del Golf,Medicina General,109,68.5,3658.73,53.41
PMM Tocumen,Oftalmología,70,43.25,3723.93,86.1


### N5: Análisis de Citas por Jornada y Hora

In [0]:
%sql
CREATE OR REPLACE TABLE citas_jornadas AS
SELECT 
    nom_sucursal AS Sucursal,
    especialidad_medica AS Especialidad,
    CASE 
        -- REGEXP_EXTRACT busca la primera coincidencia de hora (ej: "09:30") y extrae solo los primeros 2 dígitos (\d{2})
        WHEN CAST(REGEXP_EXTRACT(hr_inicio_cit, '(\\d{2}):\\d{2}', 1) AS INT) < 12 THEN 'Mañana (07:00 - 11:59)'
        WHEN CAST(REGEXP_EXTRACT(hr_inicio_cit, '(\\d{2}):\\d{2}', 1) AS INT) BETWEEN 12 AND 16 THEN 'Tarde (12:00 - 16:59)'
        ELSE 'Noche (17:00 - 20:00)'
    END AS Jornada_Horaria,
    COUNT(*) AS Cantidad_Citas,
    ROUND((COUNT(*) / (SELECT COUNT(*) FROM v_citas_pmm)) * 100, 2) AS Porcentaje_Demanda_PCT
FROM v_citas_pmm
GROUP BY 
    CASE 
        WHEN CAST(REGEXP_EXTRACT(hr_inicio_cit, '(\\d{2}):\\d{2}', 1) AS INT) < 12 THEN 'Mañana (07:00 - 11:59)'
        WHEN CAST(REGEXP_EXTRACT(hr_inicio_cit, '(\\d{2}):\\d{2}', 1) AS INT) BETWEEN 12 AND 16 THEN 'Tarde (12:00 - 16:59)'
        ELSE 'Noche (17:00 - 20:00)'
    END, 
    nom_sucursal, 
    especialidad_medica;
SELECT * FROM citas_jornadas;

Sucursal,Especialidad,Jornada_Horaria,Cantidad_Citas,Porcentaje_Demanda_PCT
PMM Brisas del Golf,Otorrinolaringología,Tarde (12:00 - 16:59),52,0.52
PMM Costa del Este,Medicina General,Tarde (12:00 - 16:59),124,1.24
PMM Costa del Este,Nefrología,Mañana (07:00 - 11:59),71,0.71
PMM Costa del Este,Gastroenterología,Tarde (12:00 - 16:59),69,0.69
PMM Tocumen,Ginecología,Mañana (07:00 - 11:59),13,0.13
PMM Tocumen,Gastroenterología,Tarde (12:00 - 16:59),31,0.31
PMM San Francisco,Oftalmología,Tarde (12:00 - 16:59),122,1.22
PMM San Francisco,Fisioterapia,Tarde (12:00 - 16:59),94,0.94
PMM San Francisco,Otorrinolaringología,Tarde (12:00 - 16:59),134,1.34
PMM San Francisco,Pediatría,Mañana (07:00 - 11:59),41,0.41


### N6: Reembolso Promedio de Aseguradoras por Especialidad

In [0]:
%sql
CREATE OR REPLACE TABLE citas_especialidad AS
SELECT 
    nom_sucursal AS Sucursal,
    especialidad_medica AS Especialidad,
    COUNT(*) AS Citas_Aseguradas,
    ROUND(SUM(pago_aseg), 2) AS Facturacion_A_Seguros_USD,
    ROUND(AVG(pago_aseg), 2) AS Reembolso_Promedio_Aseguradora_USD
FROM v_citas_pmm
WHERE estado_cita = 'Completada' AND nom_aseguradora != 'Sin Seguro'
GROUP BY especialidad_medica, nom_sucursal;
SELECT * FROM citas_especialidad;

Sucursal,Especialidad,Citas_Aseguradas,Facturacion_A_Seguros_USD,Reembolso_Promedio_Aseguradora_USD
PMM San Francisco,Radiología,215,10821.53,50.33
PMM Costa del Este,Pediatría,55,1670.39,30.37
PMM San Francisco,Urología,88,3226.6,36.67
PMM El Dorado,Otorrinolaringología,140,5321.29,38.01
PMM San Francisco,Gastroenterología,127,6905.59,54.37
PMM San Francisco,Nefrología,123,7042.54,57.26
PMM Tocumen,Otorrinolaringología,73,2314.82,31.71
PMM Brisas del Golf,Cardiología,49,1951.37,39.82
PMM Brisas del Golf,Medicina General,109,1861.92,17.08
PMM Tocumen,Oftalmología,70,1650.02,23.57
